<a href="https://colab.research.google.com/github/achinthyam08/AI-LAB/blob/main/1BF24CS014_week8_FOL_Resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
def is_variable(term):
    return term[0].islower()



def dereference(term, subst):
    while term in subst:
        term = subst[term]
    return term



def unify(t1, t2, subst):

    t1 = dereference(t1, subst)
    t2 = dereference(t2, subst)

    if t1 == t2:
        return True

    if is_variable(t1):
        subst[t1] = t2
        return True

    if is_variable(t2):
        subst[t2] = t1
        return True

    return False



def apply_substitution(literal, subst):

    sign, pred, args = literal

    new_args = [
        dereference(arg, subst) for arg in args
    ]

    return (sign, pred, new_args)



def standardize(clause, prefix):

    mapping = {}

    new_clause = []

    for sign, pred, args in clause:

        new_args = []

        for arg in args:

            if is_variable(arg):

                if arg not in mapping:
                    mapping[arg] = prefix + arg

                new_args.append(mapping[arg])

            else:
                new_args.append(arg)

        new_clause.append((sign, pred, new_args))

    return new_clause



count = 0
clause_registry = {}

def register_clause(clause, name, parents=None):
    clause_registry[name] = {
        'clause': clause,
        'parents': parents
    }
    return clause

def resolve(c1_name, c2_name):

    global count
    count += 1

    c1_data = clause_registry[c1_name]
    c2_data = clause_registry[c2_name]

    c1 = standardize(c1_data['clause'], "a" + str(count) + "_")
    c2 = standardize(c2_data['clause'], "b" + str(count) + "_")

    for i, l1 in enumerate(c1):

        for j, l2 in enumerate(c2):

            sign1, pred1, args1 = l1
            sign2, pred2, args2 = l2


            if sign1 != sign2 and pred1 == pred2:

                if len(args1) != len(args2):
                    continue

                subst = {}


                success = True

                for t1, t2 in zip(args1, args2):

                    if not unify(t1, t2, subst):
                        success = False
                        break

                if success:


                    remaining = (
                        c1[:i] + c1[i+1:] +
                        c2[:j] + c2[j+1:]
                    )


                    result = [
                        apply_substitution(lit, subst)
                        for lit in remaining
                    ]

                    resolved_name = f"R{count}"
                    register_clause(result, resolved_name, (c1_name, c2_name))
                    return resolved_name

    return None



def display_by_name(name):
    clause = clause_registry[name]['clause']
    if not clause:
        return f"{name}: EMPTY CLAUSE (□)"

    result = []
    for sign, pred, args in clause:
        text = pred + "(" + ",".join(args) + ")"
        if not sign:
            text = "NOT " + text
        result.append(text)

    return f"{name}: " + " OR ".join(result)



def print_resolution_tree(node_name, depth=0, visited=None):
    if visited is None:
        visited = set()

    indent = "    " * depth
    node_info = clause_registry[node_name]
    parents = node_info['parents']

    if parents:
        print(f"{indent}├── {display_by_name(node_name)} [Resolved from {parents[0]} and {parents[1]}]")
        if node_name not in visited:
            visited.add(node_name)
            print_resolution_tree(parents[0], depth + 1, visited)
            print_resolution_tree(parents[1], depth + 1, visited)
    else:
        print(f"{indent}└── {display_by_name(node_name)} [Axiom]")




register_clause([(False, "food", ["x"]), (True, "likes", ["John", "x"])], "C1")
register_clause([(False, "eats", ["x", "y"]), (True, "killed", ["x"]), (True, "food", ["y"])], "C2")
register_clause([(True, "eats", ["Anil", "Peanuts"])], "C3")
register_clause([(True, "alive", ["Anil"])], "C4")
register_clause([(False, "alive", ["x"]), (False, "killed", ["x"])], "C5")
register_clause([(False, "likes", ["John", "Peanuts"])], "C6")



print("RESOLUTION STEPS:\n")


R1 = resolve("C4", "C5")
print(display_by_name(R1))


R2 = resolve("C2", "C3")
print(display_by_name(R2))


R3 = resolve(R2, R1)
print(display_by_name(R3))


R4 = resolve("C1", R3)
print(display_by_name(R4))


R5 = resolve(R4, "C6")
print(display_by_name(R5))


print("\n=========================================")
print("COMPLETE RESOLUTION TREE (Contradiction Proof)")
print("=========================================\n")
print_resolution_tree(R5)


RESOLUTION STEPS:

R1: NOT killed(Anil)
R2: killed(Anil) OR food(Peanuts)
R3: food(Peanuts)
R4: likes(John,Peanuts)
R5: EMPTY CLAUSE (□)

COMPLETE RESOLUTION TREE (Contradiction Proof)

├── R5: EMPTY CLAUSE (□) [Resolved from R4 and C6]
    ├── R4: likes(John,Peanuts) [Resolved from C1 and R3]
        └── C1: NOT food(x) OR likes(John,x) [Axiom]
        ├── R3: food(Peanuts) [Resolved from R2 and R1]
            ├── R2: killed(Anil) OR food(Peanuts) [Resolved from C2 and C3]
                └── C2: NOT eats(x,y) OR killed(x) OR food(y) [Axiom]
                └── C3: eats(Anil,Peanuts) [Axiom]
            ├── R1: NOT killed(Anil) [Resolved from C4 and C5]
                └── C4: alive(Anil) [Axiom]
                └── C5: NOT alive(x) OR NOT killed(x) [Axiom]
    └── C6: NOT likes(John,Peanuts) [Axiom]
